# Annotate temple/devotional images with Claude (vision)

Runs the structured annotation prompt against each image via the Claude API and saves the
result as JSONL -- one line per image. This is a paid API call per image (unlike the
self-hosted CLIP/BLIP notebooks), so:

- Test on a small `SAMPLE_N` first, check the output quality and your Anthropic console's
  actual usage/cost, before running the full set.
- Default model is `claude-haiku-4-5-20251001` (cheaper) -- switch `MODEL` to
  `claude-sonnet-5` if Haiku's annotations aren't reliable enough on this schema.
- The API key is entered via `getpass` (never printed, never saved in a cell's
  source/output) -- do not paste your key directly into a code cell, and don't commit this
  notebook to git if you ever change that.

In [ ]:
!pip install -q anthropic

In [ ]:
import getpass
import anthropic

api_key = getpass.getpass("Anthropic API key: ")
client = anthropic.Anthropic(api_key=api_key)

MODEL = "claude-haiku-4-5-20251001"  # swap to "claude-sonnet-5" if quality isn't enough

## 1. Point at your images
Upload a zip and unzip it, or mount Drive and point `IMAGE_DIR` there.

In [ ]:
# EDIT ME
IMAGE_DIR = "/content/images"

import os
image_paths = [
    os.path.join(IMAGE_DIR, f) for f in sorted(os.listdir(IMAGE_DIR))
    if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
] if os.path.isdir(IMAGE_DIR) else []
print(f"{len(image_paths)} images found in {IMAGE_DIR}")

## 2. Annotation prompt + call function

In [ ]:
ANNOTATION_PROMPT = """You are an expert visual annotator building a dataset of Indian temples.

Analyze the provided image carefully.

Extract ONLY information that can be supported by visible evidence.

Do NOT guess or hallucinate:
- temple name
- deity name
- location
- historical facts
- mythology
- festival name
- religious tradition

If something cannot be determined from the image, use null or an empty list.

Return ONLY valid JSON using this exact schema:

{
  "image_description": "",
  "temple_name": null,
  "deity": null,
  "architecture": {
    "style": [],
    "materials": [],
    "roof": [],
    "tower": [],
    "features": []
  },
  "structures": [],
  "sculptures": [],
  "religious_symbols": [],
  "rituals": [],
  "festival_activity": [],
  "people": [],
  "environment": [],
  "visible_text": [],
  "colors": [],
  "objects": [],
  "visual_tags": [],
  "short_caption": "",
  "detailed_caption": "",
  "confidence": 0.0
}

For every observation, prioritize visible evidence.

Architecture:
Identify visible characteristics such as:
- gopuram
- vimana
- mandapa
- pillars
- stone construction
- wooden construction
- tiled roof
- carvings
- arches
- entrance structures
- walls
- courtyard

Structures:
List important visible temple structures.

Sculptures:
Describe visible statues, carvings, animals, motifs and decorative elements.

Religious symbols:
Describe only symbols that are actually visible.

Rituals and festival activity:
Describe what people are visibly doing.
Do not identify a festival unless there is clear visual evidence.

Visible text:
Transcribe readable text exactly as seen.
Do not correct or invent text.

Visual tags:
Generate 5–15 concise tags describing the visible content.
Do not use guessed temple names or locations.

short_caption:
One accurate sentence.

detailed_caption:
A detailed 30–80 word description of the image.

confidence:
Overall confidence from 0.0 to 1.0.

Return JSON only."""

In [ ]:
import base64
import json
import re
import time
from io import BytesIO
from PIL import Image

MAX_EDGE = 1568  # Claude vision has diminishing returns above this; keeps requests smaller/cheaper

def _prep_image_b64(path):
    img = Image.open(path).convert("RGB")
    if max(img.size) > MAX_EDGE:
        img.thumbnail((MAX_EDGE, MAX_EDGE))
    buf = BytesIO()
    img.save(buf, format="JPEG", quality=90)
    return base64.standard_b64encode(buf.getvalue()).decode("utf-8")

def _extract_json(text):
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if match:
            return json.loads(match.group(0))
        raise

def annotate_image(path, max_retries=3):
    img_b64 = _prep_image_b64(path)
    for attempt in range(max_retries):
        try:
            resp = client.messages.create(
                model=MODEL,
                max_tokens=1500,
                messages=[{
                    "role": "user",
                    "content": [
                        {"type": "image", "source": {"type": "base64", "media_type": "image/jpeg", "data": img_b64}},
                        {"type": "text", "text": ANNOTATION_PROMPT},
                    ],
                }],
            )
            return _extract_json(resp.content[0].text)
        except json.JSONDecodeError as e:
            print(f"  unparseable JSON for {path} (attempt {attempt+1}): {e}")
        except Exception as e:
            print(f"  API error for {path} (attempt {attempt+1}): {e}")
            time.sleep(2 ** attempt)
    return None

## 3. Run it -- test on a small sample first

In [ ]:
SAMPLE_N = 5  # raise this only after checking the output below looks right and cost is acceptable
OUTPUT_PATH = "annotations.jsonl"

with open(OUTPUT_PATH, "a") as out_f:
    for path in image_paths[:SAMPLE_N]:
        result = annotate_image(path)
        if result is None:
            print(f"skipped (failed after retries): {path}")
            continue
        record = {"file": os.path.basename(path), **result}
        out_f.write(json.dumps(record) + "\n")
        out_f.flush()  # write incrementally -- a crash partway through shouldn't lose completed work
        print(f"{os.path.basename(path)}: {result.get('short_caption')} (confidence={result.get('confidence')})")

print(f"\nSaved to {OUTPUT_PATH}")

Check `annotations.jsonl` and your Anthropic console's usage page. If the annotations look
right and the cost for `SAMPLE_N` images is acceptable scaled up to your full set, raise
`SAMPLE_N` to `len(image_paths)` and re-run (it appends, so already-annotated images in this
run won't be duplicated only if you clear/rename `annotations.jsonl` between full runs).

## 4. Next step: feed into BLIP fine-tuning
`short_caption` / `detailed_caption` per image here replace the fixed `CAPTION_TEMPLATES`
dict in `BLIP_Devotional_Finetune.ipynb` -- swap that notebook's caption lookup to read
from this `annotations.jsonl` instead, keyed by filename, for real per-image captions
instead of one repeated sentence per cluster.